# Clase 5 — EDA de morosidad por producto

## Pregunta
¿Cómo se distribuye la morosidad entre los distintos productos registrados?

### Criterio de trabajo
No modificamos el CSV original. Revisamos calidad primero. No imputamos faltantes y no normalizamos variables porque acá hacemos análisis descriptivo, no entrenamiento de un modelo.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidatas = [
    Path('../modulo_3/dataset_credito.csv'),
    Path('../data/dataset_credito.csv'),
    Path('data/dataset_credito.csv'),
]
csv_path = next((p for p in candidatas if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('No encontré dataset_credito.csv. Revisá la ruta.')

df = pd.read_csv(csv_path)
print('Archivo:', csv_path)
print('Filas:', df.shape[0])
print('Columnas:', df.shape[1])

## 1. Calidad de datos
Revisamos tipos, faltantes y duplicados. Un faltante no se convierte automáticamente en cero.

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())
print('\nFaltantes:')
print(df.isna().sum()[df.isna().sum() > 0])

if 'edad' in df.columns:
    print('\nEdades potencialmente atípicas (<18 o >100):', ((df['edad'] < 18) | (df['edad'] > 100)).sum())

if 'BCRA_Peor_Situacion' in df.columns:
    print('Valores BCRA negativos:', (df['BCRA_Peor_Situacion'] < 0).sum())

if 'Nivel_Socioeconomico' in df.columns:
    print('Valores Nivel_Socioeconomico negativos:', (df['Nivel_Socioeconomico'] < 0).sum())

## 2. Morosidad general
Primero calculamos el porcentaje general para tener una referencia.

In [ ]:
morosidad_general = df['moroso'].mean() * 100
print(f'Morosidad general: {morosidad_general:.2f}%')

## 3. Morosidad por producto
Importante: miramos cantidad de casos y proporción de morosos. Un producto puede tener más morosos simplemente porque tiene más registros.

In [ ]:
por_producto = (
    df.groupby('PRODUCTO')
      .agg(
          casos=('moroso', 'size'),
          morosos=('moroso', 'sum'),
          proporcion_morosos=('moroso', 'mean'),
      )
      .sort_values('proporcion_morosos', ascending=False)
)
por_producto['proporcion_morosos_pct'] = por_producto['proporcion_morosos'] * 100
por_producto[['casos', 'morosos', 'proporcion_morosos_pct']].round({'proporcion_morosos_pct': 2})

In [ ]:
ax = por_producto['proporcion_morosos_pct'].plot(kind='bar', figsize=(10,5))
ax.set_title('Proporción de morosidad por producto')
ax.set_xlabel('Producto')
ax.set_ylabel('Morosos (%)')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Hallazgo y límite
Se observan diferencias importantes en la proporción de morosos entre productos. Esto es útil para decidir qué segmentos conviene investigar.

**Límite:** el análisis es descriptivo. No demuestra causalidad y algunos productos tienen pocos registros. Tampoco conocemos, a partir del CSV solamente, todo el contexto de origen, período y definición operativa de las variables.

## 5. Próxima pregunta
¿La diferencia de morosidad entre productos se mantiene cuando también consideramos ingresos, edad, nivel socioeconómico, deuda y días de atraso?